# BraTS 2021 - Lightweight **Inverted-Residual + SE + Attention** 2D U-Net (complete notebook)

Brain-tumor segmentation with an **Attention U-Net** whose **encoder uses inverted residual blocks with Squeeze-and-Excitation (SE)**, a **Dice-only loss**, the **Adamax** optimizer, early stopping, data augmentation, and full metrics.

Pipeline: find data -> build 2D slices (tumor + neighbor context) -> **train / validate / test** -> report **Dice, IoU, Precision, Recall, HD95** (WT / TC / ET) -> visualise predictions.

Built for a laptop with an **RTX 5070** and a **Miniconda** virtual environment. Every section prints or plots an output.

**BraTS labels:** 1 = necrotic core, 2 = edema, 4 = enhancing tumor (3 unused).
**Regions:** `WT = 1,2,4`, `TC = 1,4`, `ET = 4`.
**This version:** inverted-residual + SE encoder, attention-gate decoder, Adamax optimizer, up to 100 epochs (early stopping patience 25), 350 patients (210 train / 70 val / 70 test), LR 1e-4, Dice loss only, float32.

## 0. Miniconda virtual environment - full setup (do once, in a terminal)

Your **RTX 5070** is Blackwell (compute capability **sm_120**). The default PyTorch build only has kernels up to sm_90, so it crashes with *"no kernel image is available"* the moment training starts. You need the **CUDA 12.8 (cu128)** build.

Open **Anaconda Prompt (miniconda3)** and run:

```bat
conda create -n brats python=3.11 -y
conda activate brats

:: PyTorch for the RTX 5070 (sm_120):
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu128

:: the rest (scipy is needed for HD95):
pip install nibabel opencv-python matplotlib tqdm torchinfo scipy notebook ipykernel

:: register this env as a Jupyter kernel:
python -m ipykernel install --user --name brats --display-name "Python (brats)"
```

If the GPU check (Section 3) fails with an sm_120 error, switch to the nightly cu128 build:

```bat
pip uninstall -y torch torchvision
pip install --pre torch torchvision --index-url https://download.pytorch.org/whl/nightly/cu128 --no-cache-dir
```

## 1. Imports
**Output:** installed library versions.

In [ ]:
import os, glob, random
import numpy as np
import nibabel as nib
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

try:
    from torchinfo import summary as torchinfo_summary
    HAS_TORCHINFO = True
except Exception:
    HAS_TORCHINFO = False

print("numpy   :", np.__version__)
print("torch   :", torch.__version__)
print("opencv  :", cv2.__version__)
print("nibabel :", nib.__version__)
print("torchinfo available:", HAS_TORCHINFO)

## 2. Config
**Patients:** 350 total = 210 train / 70 val / 70 test (60/20/20). **LR 1e-4, Dice loss, float32, batch 16, up to 100 epochs.**

In [ ]:
DATA_ROOTS = [
    r"D:\archive",       # your laptop (D: drive)
    "/kaggle/input",     # Kaggle datasets
]
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "./brats_out"

NUM_PATIENTS = 350          # total patients to use
N_VAL_PATIENTS = 70         # validation patients (20%)
N_TEST_PATIENTS = 70        # test patients (20%)
#   -> train patients = 350 - 70 - 70 = 210 (60%)

IMG_SIZE = 128              # slices resized to 128x128
NEIGHBOR_SLICES = 2         # keep 2 slices before & after each tumor slice (context)
KEEP_EMPTY_FRACTION = 0.10  # small fraction of purely-empty slices for balance
MAX_SLICES_PER_PATIENT = 100  # cap per patient (set None to keep ALL valid slices)
MIN_BRAIN_FRACTION = 0.02   # skip near-black slices
USE_FLOAT32 = True          # True = float32 (more RAM, no precision loss); False = float16

EPOCHS = 100
PATIENCE = 25               # early stopping: stop if val Dice doesn't improve for 25 epochs
BATCH_SIZE = 16
LR = 1e-4                   # 0.0001
WEIGHT_DECAY = 1e-4
BASE_FILTERS = 32           # 32 -> lightweight encoder
SEED = 42

MODALITIES = ["flair", "t1", "t1ce", "t2"]
REGIONS = ["WT", "TC", "ET"]
IMG_DTYPE = np.float32 if USE_FLOAT32 else np.float16

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
os.makedirs(OUT_DIR, exist_ok=True)

print(f"Patients: {NUM_PATIENTS} total -> {NUM_PATIENTS-N_VAL_PATIENTS-N_TEST_PATIENTS} train / "
      f"{N_VAL_PATIENTS} val / {N_TEST_PATIENTS} test")
print(f"img={IMG_SIZE} | batch={BATCH_SIZE} | epochs={EPOCHS} (patience {PATIENCE}) | LR={LR} | "
      f"dtype={'float32' if USE_FLOAT32 else 'float16'} | max_slices/patient={MAX_SLICES_PER_PATIENT}")
print("Loss = Dice loss only | Optimizer = Adamax | Encoder = Inverted-Residual (+SE) | Decoder = Attention gates")
print("Output folder:", OUT_DIR)

## 3. GPU check
Runs a real matrix multiply - the only reliable test that the RTX 5070 build works.
**Output:** GPU name, compute capability, pass/fail.

In [ ]:
def get_device():
    if not torch.cuda.is_available():
        print("[warn] CUDA not available -> CPU (slow).")
        return torch.device("cpu")
    print("GPU:", torch.cuda.get_device_name(0),
          "(sm_%d%d)" % torch.cuda.get_device_capability(0))
    try:
        _ = torch.randn(1024, 1024, device="cuda") @ torch.randn(1024, 1024, device="cuda")
        torch.cuda.synchronize()
        print("GPU compute test: OK")
    except RuntimeError as e:
        raise SystemExit("GPU compute failed - install the cu128 build (Section 0).\n" + str(e))
    return torch.device("cuda")

device = get_device()

## 4. Find the data
Searches recursively for `*_flair.nii.gz`. Unzip `BraTS2021_Training_Data.zip` first.
**Output:** how many patients were found.

In [ ]:
def find_patient_dirs(roots):
    for root in roots:
        if not os.path.isdir(root):
            continue
        flairs = glob.glob(os.path.join(root, "**", "*_flair.nii.gz"), recursive=True)
        dirs = sorted({os.path.dirname(f) for f in flairs})
        if dirs:
            print(f"Found {len(dirs)} patients under {root}")
            return dirs
    raise SystemExit("No patients found. Unzip the data or edit DATA_ROOTS.")

patients = find_patient_dirs(DATA_ROOTS)
print("Example patient folder:", os.path.basename(patients[0]))

## 5. Preprocessing functions (improved slice selection)
For each patient we now keep:
1. **All tumor slices**, plus
2. **NEIGHBOR_SLICES before & after** each tumor slice (anatomical context), plus
3. a **small fraction of empty slices** for balance (tumor + context are prioritised).

Images are stored as **float32** (no precision loss). Up to `MAX_SLICES_PER_PATIENT` per patient (tumor slices are kept first).
**Output:** shapes for one test patient.

In [ ]:
def znorm(vol):
    brain = vol > 0
    if brain.sum() == 0:
        return vol
    vol = (vol - vol[brain].mean()) / (vol[brain].std() + 1e-8)
    vol[~brain] = 0.0
    return vol

def load_vol(pdir, modality):
    pid = os.path.basename(pdir)
    return nib.load(os.path.join(pdir, f"{pid}_{modality}.nii.gz")).get_fdata().astype(np.float32)

def rz_img(sl):
    return cv2.resize(sl, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)

def rz_msk(sl):
    return cv2.resize(sl.astype(np.uint8), (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)

def select_slice_indices(seg, mods, Z):
    brain_ok = [(mods[0][:, :, z] != 0).mean() >= MIN_BRAIN_FRACTION for z in range(Z)]
    tumor = [z for z in range(Z) if brain_ok[z] and seg[:, :, z].max() > 0]
    ctx = set()
    for tz in tumor:                                   # add neighbours for context
        for dz in range(-NEIGHBOR_SLICES, NEIGHBOR_SLICES + 1):
            zz = tz + dz
            if 0 <= zz < Z and brain_ok[zz]:
                ctx.add(zz)
    ctx -= set(tumor)
    used = set(tumor) | ctx
    empties = [z for z in range(Z) if brain_ok[z] and z not in used]
    random.shuffle(empties)
    empties = empties[:int(len(empties) * KEEP_EMPTY_FRACTION)]
    keep = list(tumor) + sorted(ctx) + empties          # priority: tumor > context > empty
    if MAX_SLICES_PER_PATIENT and len(keep) > MAX_SLICES_PER_PATIENT:
        keep = keep[:MAX_SLICES_PER_PATIENT]            # tumor slices kept first
    return sorted(set(keep))

def build_patient_slices(pdir):
    mods = [znorm(load_vol(pdir, m)) for m in MODALITIES]
    seg = load_vol(pdir, "seg")
    Z = seg.shape[2]
    keep = select_slice_indices(seg, mods, Z)
    imgs, masks = [], []
    for z in keep:
        img = np.stack([rz_img(m[:, :, z]) for m in mods], 0).astype(IMG_DTYPE)
        s = seg[:, :, z]
        mask = np.stack([rz_msk(np.isin(s, [1, 2, 4])),
                         rz_msk(np.isin(s, [1, 4])),
                         rz_msk(s == 4)], 0).astype(np.uint8)
        imgs.append(img); masks.append(mask)
    if not imgs:
        return None, None
    return np.stack(imgs), np.stack(masks)

_img, _msk = build_patient_slices(patients[0])
print("One patient -> images", _img.shape, _img.dtype, "| masks", _msk.shape)
print("(N_slices, 4 modalities, H, W) and (N_slices, 3 regions, H, W)")

## 6. Build train / val / test arrays (split by patient count)
**Output:** patient + slice counts per split and RAM used.

In [ ]:
def build_dataset(patient_dirs):
    patient_dirs = list(patient_dirs)
    random.shuffle(patient_dirs)
    patient_dirs = patient_dirs[:NUM_PATIENTS]
    val_set = set(patient_dirs[:N_VAL_PATIENTS])
    test_set = set(patient_dirs[N_VAL_PATIENTS:N_VAL_PATIENTS + N_TEST_PATIENTS])

    buckets = {"train": [[], []], "val": [[], []], "test": [[], []]}
    pcount = {"train": 0, "val": 0, "test": 0}
    for pdir in tqdm(patient_dirs, desc="Slicing patients"):
        imgs, masks = build_patient_slices(pdir)
        if imgs is None:
            continue
        split = "val" if pdir in val_set else "test" if pdir in test_set else "train"
        buckets[split][0].append(imgs); buckets[split][1].append(masks)
        pcount[split] += 1

    out = {}
    for k, (xs, ys) in buckets.items():
        out[k] = (np.concatenate(xs), np.concatenate(ys))
    return out, pcount

data, pcount = build_dataset(patients)
(Xtr, Ytr), (Xva, Yva), (Xte, Yte) = data["train"], data["val"], data["test"]
ram = (Xtr.nbytes + Xva.nbytes + Xte.nbytes) / 1e9
print(f"TRAIN: {pcount['train']:3d} patients -> {len(Xtr):6d} slices")
print(f"VAL  : {pcount['val']:3d} patients -> {len(Xva):6d} slices")
print(f"TEST : {pcount['test']:3d} patients -> {len(Xte):6d} slices")
print(f"Images in RAM: ~{ram:.1f} GB ({Xtr.dtype})")
print("If you hit a MemoryError: lower MAX_SLICES_PER_PATIENT (e.g. 70) or set USE_FLOAT32=False, then re-run from Section 2.")

## 6b. Peek at one preprocessed slice
**Output:** the 4 modalities and the tumor mask for a validation slice.

In [ ]:
def show_sample(X, Y):
    idx = next((i for i in range(len(Y)) if Y[i, 0].sum() > 50), 0)
    fig, ax = plt.subplots(1, 5, figsize=(15, 3))
    for j, name in enumerate(MODALITIES):
        ax[j].imshow(X[idx, j].astype(np.float32), cmap="gray")
        ax[j].set_title(name); ax[j].axis("off")
    ax[4].imshow(X[idx, 0].astype(np.float32), cmap="gray")
    ax[4].imshow(Y[idx, 0], alpha=0.4, cmap="Reds")
    ax[4].set_title("WT mask"); ax[4].axis("off")
    plt.tight_layout(); plt.show()

show_sample(Xva, Yva)

## 6c. Tumor ratio analysis
How much of the data is actually tumor versus background - the class imbalance that makes segmentation hard and justifies the Dice loss.
**Output:** % of slices containing tumor, pixel-level tumor fraction per region (with the background:foreground ratio), and coverage histograms.

In [ ]:
def tumor_ratio_analysis(splits):
    HW = IMG_SIZE * IMG_SIZE
    print("Slice-level: how many slices contain tumor (WT > 0)")
    print("Split |  slices | tumor slices | % tumor")
    print("------+---------+--------------+--------")
    tot = {r: [0.0, 0.0] for r in REGIONS}
    cover = {r: [] for r in REGIONS}
    for name, Y in splits.items():
        Yf = Y.astype(np.float32)
        wt = Yf[:, 0].reshape(len(Yf), -1).sum(1)
        nt = int((wt > 0).sum())
        print(f"{name:5s} | {len(Yf):7d} | {nt:12d} | {100*nt/len(Yf):5.1f}%")
        for ci, r in enumerate(REGIONS):
            tot[r][0] += Yf[:, ci].sum(); tot[r][1] += Yf[:, ci].size
            per = Yf[:, ci].reshape(len(Yf), -1).sum(1)
            cover[r].extend((per[per > 0] / HW * 100).tolist())

    print("\nPixel-level: fraction of ALL pixels that are tumor (class imbalance)")
    print("Region | tumor % | background : foreground")
    print("-------+---------+------------------------")
    for r in REGIONS:
        frac = tot[r][0] / tot[r][1]
        ratio = (1 - frac) / frac if frac > 0 else float("inf")
        print(f"{r:6s} | {100*frac:6.3f}% | {ratio:8.1f} : 1")

    print("\nAmong tumor-containing slices: mean tumor area (% of slice)")
    for r in REGIONS:
        c = cover[r]
        if c:
            print(f"  {r}: mean {np.mean(c):5.2f}%  median {np.median(c):5.2f}%  max {np.max(c):5.2f}%")

    fig, ax = plt.subplots(1, 3, figsize=(13, 3.5))
    for i, r in enumerate(REGIONS):
        if cover[r]:
            ax[i].hist(cover[r], bins=30, color="tab:red", alpha=0.7)
        ax[i].set_title(f"{r} coverage per tumor slice"); ax[i].set_xlabel("% of slice"); ax[i].set_ylabel("slices")
    plt.tight_layout(); plt.show()

tumor_ratio_analysis({"train": Ytr, "val": Yva, "test": Yte})

## 7. Dataset + DataLoaders (with augmentation)
Augmentation (random flips, 90-degree rotations, small intensity jitter) is applied to the **training set only** to reduce overfitting. Val/test stay clean.
**Output:** shape of one training batch.

In [ ]:
class SliceDS(Dataset):
    def __init__(self, X, Y, augment=False):
        self.X, self.Y, self.augment = X, Y, augment
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        x = self.X[i].astype(np.float32)   # (4,H,W)
        y = self.Y[i].astype(np.float32)   # (3,H,W)
        if self.augment:
            if random.random() < 0.5: x = x[:, :, ::-1]; y = y[:, :, ::-1]          # horizontal flip
            if random.random() < 0.5: x = x[:, ::-1, :]; y = y[:, ::-1, :]          # vertical flip
            k = random.randint(0, 3)
            if k: x = np.rot90(x, k, axes=(1, 2)); y = np.rot90(y, k, axes=(1, 2))  # 90-deg rotation
            x = x * random.uniform(0.9, 1.1) + random.uniform(-0.1, 0.1)            # intensity (image only)
            x, y = np.ascontiguousarray(x), np.ascontiguousarray(y)
        return torch.from_numpy(x.copy()), torch.from_numpy(y.copy())

pin = device.type == "cuda"
train_loader = DataLoader(SliceDS(Xtr, Ytr, augment=True),  batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=pin)
val_loader   = DataLoader(SliceDS(Xva, Yva, augment=False), batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=pin)
test_loader  = DataLoader(SliceDS(Xte, Yte, augment=False), batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=pin)

xb, yb = next(iter(train_loader))
print("Batch images:", tuple(xb.shape), "| Batch masks:", tuple(yb.shape))
print("Augmentation ON for training, OFF for val/test")

## 8. Inverted-Residual + SE + Attention U-Net + its internal architecture
The encoder is built from **inverted residual blocks** (MobileNet/EfficientNet style), each with an **SE unit** inside:
- **expand** (1x1 conv widens the channels) -> **depthwise 3x3 conv** (cheap spatial filtering) -> **SE** (re-weight channels) -> **project** (1x1 conv back down, linear) -> **residual** add.

This keeps the encoder lightweight while still learning strong features. The decoder keeps **attention gates** on every skip connection (spatial focus on tumor regions).
So the model combines: inverted-residual efficiency + SE channel attention + spatial attention gates.
**Output:** the full module tree, a layer-by-layer summary, and total parameters.

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(True))
    def forward(self, x):
        return self.net(x)

class SEBlock(nn.Module):
    """Squeeze-and-Excitation: re-weight channels by their global importance."""
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(channels // reduction, 4)
        self.pool = nn.AdaptiveAvgPool2d(1)                 # squeeze: global avg pool
        self.fc = nn.Sequential(                            # excitation: FC -> ReLU -> FC -> Sigmoid
            nn.Linear(channels, hidden, bias=False), nn.ReLU(inplace=True),
            nn.Linear(hidden, channels, bias=False), nn.Sigmoid())
    def forward(self, x):
        b, c, _, _ = x.shape
        s = self.pool(x).view(b, c)
        s = self.fc(s).view(b, c, 1, 1)
        return x * s                                        # scale each channel

class InvertedResidualSE(nn.Module):
    """MobileNet-style inverted residual block with an SE unit (encoder block).
    expand (1x1) -> depthwise 3x3 -> SE -> project (1x1, linear) + residual shortcut."""
    def __init__(self, cin, cout, expand_ratio=4, reduction=8):
        super().__init__()
        hidden = cin * expand_ratio
        self.expand = nn.Sequential(
            nn.Conv2d(cin, hidden, 1, bias=False), nn.BatchNorm2d(hidden), nn.ReLU6(True))
        self.depthwise = nn.Sequential(
            nn.Conv2d(hidden, hidden, 3, padding=1, groups=hidden, bias=False),
            nn.BatchNorm2d(hidden), nn.ReLU6(True))
        self.se = SEBlock(hidden, reduction)                       # SE on the expanded features
        self.project = nn.Sequential(
            nn.Conv2d(hidden, cout, 1, bias=False), nn.BatchNorm2d(cout))   # linear projection
        self.shortcut = (nn.Identity() if cin == cout else
                         nn.Sequential(nn.Conv2d(cin, cout, 1, bias=False), nn.BatchNorm2d(cout)))
    def forward(self, x):
        out = self.project(self.se(self.depthwise(self.expand(x))))
        return out + self.shortcut(x)                              # inverted residual connection

class AttentionGate(nn.Module):
    """Weights the skip features x using the decoder gating signal g (same spatial size)."""
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(nn.Conv2d(F_g, F_int, 1), nn.BatchNorm2d(F_int))
        self.W_x = nn.Sequential(nn.Conv2d(F_l, F_int, 1), nn.BatchNorm2d(F_int))
        self.psi = nn.Sequential(nn.Conv2d(F_int, 1, 1), nn.BatchNorm2d(1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)
    def forward(self, g, x):
        a = self.relu(self.W_g(g) + self.W_x(x))
        a = self.psi(a)          # attention coefficients in [0,1]
        return x * a             # keep tumor-relevant skip features

class IRSEAttentionUNet2D(nn.Module):
    def __init__(self, in_ch=4, out_ch=3, base=32):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        # encoder: inverted-residual (+SE) block at every stage
        self.d1 = InvertedResidualSE(in_ch, base)
        self.d2 = InvertedResidualSE(base, base * 2)
        self.d3 = InvertedResidualSE(base * 2, base * 4)
        self.d4 = InvertedResidualSE(base * 4, base * 8)
        self.bott = DoubleConv(base * 8, base * 16)          # bottleneck (plain)
        self.up4 = nn.ConvTranspose2d(base * 16, base * 8, 2, stride=2)
        self.att4 = AttentionGate(base * 8, base * 8, base * 4); self.u4 = DoubleConv(base * 16, base * 8)
        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.att3 = AttentionGate(base * 4, base * 4, base * 2); self.u3 = DoubleConv(base * 8, base * 4)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.att2 = AttentionGate(base * 2, base * 2, base);     self.u2 = DoubleConv(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.att1 = AttentionGate(base, base, base // 2);        self.u1 = DoubleConv(base * 2, base)
        self.out = nn.Conv2d(base, out_ch, 1)
    def forward(self, x):
        c1 = self.d1(x)
        c2 = self.d2(self.pool(c1))
        c3 = self.d3(self.pool(c2))
        c4 = self.d4(self.pool(c3))
        b = self.bott(self.pool(c4))
        g4 = self.up4(b); x = self.u4(torch.cat([g4, self.att4(g4, c4)], 1))
        g3 = self.up3(x); x = self.u3(torch.cat([g3, self.att3(g3, c3)], 1))
        g2 = self.up2(x); x = self.u2(torch.cat([g2, self.att2(g2, c2)], 1))
        g1 = self.up1(x); x = self.u1(torch.cat([g1, self.att1(g1, c1)], 1))
        return self.out(x)

model = IRSEAttentionUNet2D(in_ch=4, out_ch=3, base=BASE_FILTERS).to(device)

print("========= Inverted-Residual + SE + Attention U-Net module tree =========")
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

print("\n============ Layer-by-layer summary ============")
if HAS_TORCHINFO:
    torchinfo_summary(model, input_size=(1, 4, IMG_SIZE, IMG_SIZE),
                      col_names=("input_size", "output_size", "num_params"), depth=3)
else:
    x = torch.randn(1, 4, IMG_SIZE, IMG_SIZE, device=device)
    with torch.no_grad():
        c1 = model.d1(x); print(" enc1 :", tuple(c1.shape))
        c2 = model.d2(model.pool(c1)); print(" enc2 :", tuple(c2.shape))
        c3 = model.d3(model.pool(c2)); print(" enc3 :", tuple(c3.shape))
        c4 = model.d4(model.pool(c3)); print(" enc4 :", tuple(c4.shape))
        b = model.bott(model.pool(c4)); print(" bott :", tuple(b.shape))
        print(" out  :", tuple(model(x).shape))

## 9. Dice loss (only) + Dice metric
Loss is **pure Dice loss** (no BCE). **Output:** sanity checks - a perfect prediction gives Dice = 1.0 and Dice loss ~ 0.

In [ ]:
class DiceLoss(nn.Module):
    """Soft Dice loss over the batch, averaged across the 3 regions."""
    def forward(self, logits, y, eps=1e-6):
        probs = torch.sigmoid(logits)
        inter = (probs * y).sum((0, 2, 3))
        denom = probs.sum((0, 2, 3)) + y.sum((0, 2, 3))
        dice = (2 * inter + eps) / (denom + eps)
        return 1 - dice.mean()

@torch.no_grad()
def accumulate_dice(logits, y, inter, denom, thr=0.5):
    preds = (torch.sigmoid(logits) > thr).float()
    inter += (preds * y).sum((0, 2, 3)).cpu()
    denom += (preds.sum((0, 2, 3)) + y.sum((0, 2, 3))).cpu()
    return inter, denom

def finalize_dice(inter, denom, eps=1e-6):
    return (2 * inter + eps) / (denom + eps)

_y = (torch.rand(2, 3, 32, 32) > 0.5).float()
_perfect = (_y - 0.5) * 20.0
_i, _d = accumulate_dice(_perfect, _y, torch.zeros(3), torch.zeros(3))
print("Dice metric on perfect prediction:", [round(v, 3) for v in finalize_dice(_i, _d).tolist()], "(expect ~1.0)")
print("Dice LOSS on perfect prediction  :", round(DiceLoss()(_perfect, _y).item(), 4), "(expect ~0.0)")

## 10. Train + validate (up to 100 epochs, early stopping patience 25)
Inverted-Residual+SE+Attention U-Net, **Adamax**, Dice loss, LR 1e-4, mixed precision, augmented training. Training stops automatically if val Dice doesn't improve for 25 epochs. **Output:** per-epoch **train loss, val loss, val Dice**, plus curves. Best model saved.

In [ ]:
def train_model(model, train_loader, val_loader, device):
    torch.backends.cudnn.benchmark = True
    loss_fn = DiceLoss()
    opt = torch.optim.Adamax(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=amp)

    hist = {k: [] for k in ["train", "val", "WT", "TC", "ET"]}
    best = -1.0; best_epoch = 0; no_improve = 0
    for ep in range(1, EPOCHS + 1):
        model.train(); run = 0.0
        for x, y in tqdm(train_loader, leave=False, desc=f"epoch {ep}/{EPOCHS} [train]"):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=amp):
                loss = loss_fn(model(x), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            run += loss.item() * x.size(0)
        tr_loss = run / len(train_loader.dataset)

        model.eval(); run = 0.0
        inter, denom = torch.zeros(3), torch.zeros(3)
        with torch.no_grad():
            for x, y in tqdm(val_loader, leave=False, desc=f"epoch {ep}/{EPOCHS} [val]"):
                x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
                with torch.amp.autocast("cuda", enabled=amp):
                    logits = model(x); loss = loss_fn(logits, y)
                run += loss.item() * x.size(0)
                inter, denom = accumulate_dice(logits.float(), y, inter, denom)
        va_loss = run / len(val_loader.dataset)
        wt, tc, et = finalize_dice(inter, denom).tolist()
        mean = (wt + tc + et) / 3
        sched.step()

        for k, v in zip(hist, [tr_loss, va_loss, wt, tc, et]):
            hist[k].append(v)
        star = ""
        if mean > best:
            best = mean; best_epoch = ep; no_improve = 0
            torch.save({"model": model.state_dict(), "base": BASE_FILTERS,
                        "epoch": ep, "dice": {"WT": wt, "TC": tc, "ET": et}},
                       os.path.join(OUT_DIR, "best_model.pt"))
            star = "  <- best saved"
        else:
            no_improve += 1
            star = f"  (no improve {no_improve}/{PATIENCE})"
        print(f"Epoch {ep:02d}/{EPOCHS} | train {tr_loss:.4f} | val {va_loss:.4f} | "
              f"val Dice  WT {wt:.3f}  TC {tc:.3f}  ET {et:.3f} | mean {mean:.3f}{star}")
        if no_improve >= PATIENCE:
            print(f"\nEarly stopping at epoch {ep}: no val improvement for {PATIENCE} epochs.")
            break

    ep_range = range(1, len(hist["train"]) + 1)
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ax[0].plot(ep_range, hist["train"], label="train loss")
    ax[0].plot(ep_range, hist["val"], label="val loss")
    ax[0].set_title("Dice Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
    for r in REGIONS:
        ax[1].plot(ep_range, hist[r], label=f"val {r}")
    ax[1].set_title("Validation Dice"); ax[1].set_ylim(0, 1); ax[1].set_xlabel("epoch"); ax[1].legend()
    fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "training_curves.png"), dpi=120); plt.show()
    print(f"\nBest mean validation Dice: {best:.3f} (epoch {best_epoch})")
    return hist

history = train_model(model, train_loader, val_loader, device)

## 11. Test evaluation - Dice (held-out 60 patients)
Loads the best saved model and scores it on the **test** set.
**Output:** test Dice for WT, TC, ET.

In [ ]:
ckpt = torch.load(os.path.join(OUT_DIR, "best_model.pt"), map_location=device)
best_model = IRSEAttentionUNet2D(4, 3, ckpt["base"]).to(device)
best_model.load_state_dict(ckpt["model"]); best_model.eval()
print(f"Loaded best model from epoch {ckpt['epoch']}")

inter, denom = torch.zeros(3), torch.zeros(3)
with torch.no_grad():
    for x, y in tqdm(test_loader, desc="Testing"):
        x, y = x.to(device), y.to(device)
        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            logits = best_model(x)
        inter, denom = accumulate_dice(logits.float(), y, inter, denom)
wt, tc, et = finalize_dice(inter, denom).tolist()
print("\n================ TEST Dice ================")
print(f"  WT (Whole Tumor)     : {wt:.3f}")
print(f"  TC (Tumor Core)      : {tc:.3f}")
print(f"  ET (Enhancing Tumor) : {et:.3f}")
print(f"  Mean                 : {(wt+tc+et)/3:.3f}")

## 11b. More test metrics: IoU, Precision, Recall, HD95
- **IoU (Jaccard)**, **Precision**, **Recall/Sensitivity**: higher is better (0-1).
- **HD95** (95th-percentile Hausdorff distance): boundary error, **lower is better**. In *pixels* on the 128x128 slices (approximate 2D value).

Needs `scipy`. **Output:** a per-region table of all four metrics.

In [ ]:
from scipy.ndimage import distance_transform_edt, binary_erosion

def _borders(m):
    return m ^ binary_erosion(m)

def hd95(pred, gt, spacing=(1.0, 1.0)):
    pred, gt = pred.astype(bool), gt.astype(bool)
    if not pred.any() or not gt.any():
        return np.nan
    pb, gb = _borders(pred), _borders(gt)
    dt_to_g = distance_transform_edt(~gb, sampling=spacing)
    dt_to_p = distance_transform_edt(~pb, sampling=spacing)
    d = np.hstack([dt_to_g[pb], dt_to_p[gb]])
    return np.percentile(d, 95)

best_model.eval()
TP, FP, FN = torch.zeros(3), torch.zeros(3), torch.zeros(3)
hd_lists = [[], [], []]

with torch.no_grad():
    for x, y in tqdm(test_loader, desc="Extra metrics"):
        x = x.to(device)
        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            probs = torch.sigmoid(best_model(x))
        p = (probs > 0.5).float().cpu(); g = y
        TP += (p * g).sum((0, 2, 3))
        FP += (p * (1 - g)).sum((0, 2, 3))
        FN += ((1 - p) * g).sum((0, 2, 3))
        pn, gn = p.numpy().astype(bool), g.numpy().astype(bool)
        for b in range(pn.shape[0]):
            for c in range(3):
                if pn[b, c].any() and gn[b, c].any():
                    hd_lists[c].append(hd95(pn[b, c], gn[b, c]))

eps = 1e-6
iou  = TP / (TP + FP + FN + eps)
prec = TP / (TP + FP + eps)
rec  = TP / (TP + FN + eps)

print("Region |  IoU  | Precision | Recall | HD95(px) | slices")
print("-------+-------+-----------+--------+----------+-------")
for i, r in enumerate(REGIONS):
    hd = np.mean(hd_lists[i]) if hd_lists[i] else float("nan")
    print(f"{r:6s} | {iou[i]:.3f} |   {prec[i]:.3f}   | {rec[i]:.3f} |  {hd:6.2f}  | {len(hd_lists[i])}")

## 12. Visualise test predictions
**Output:** FLAIR / ground-truth / prediction for a few test slices, with per-slice Dice.

In [ ]:
def visualize(model, X, Y, device, n_show=6):
    model.eval()
    tumor_idx = [i for i in range(len(Y)) if Y[i, 0].sum() > 50]
    picks = random.sample(tumor_idx, min(n_show, len(tumor_idx)))
    fig, axes = plt.subplots(len(picks), 3, figsize=(9, 3 * len(picks)))
    if len(picks) == 1:
        axes = axes[None, :]
    for r, i in enumerate(picks):
        x = torch.from_numpy(X[i].astype(np.float32)).unsqueeze(0).to(device)
        with torch.no_grad(), torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            pred = (torch.sigmoid(model(x))[0, 0] > 0.5).cpu().numpy()
        gt = Y[i, 0]; flair = X[i, 0].astype(np.float32)
        d = (2 * (pred * gt).sum() + 1e-6) / (pred.sum() + gt.sum() + 1e-6)
        axes[r, 0].imshow(flair, cmap="gray"); axes[r, 0].set_title("FLAIR")
        axes[r, 1].imshow(flair, cmap="gray"); axes[r, 1].imshow(gt, alpha=0.4, cmap="Reds"); axes[r, 1].set_title("Ground truth (WT)")
        axes[r, 2].imshow(flair, cmap="gray"); axes[r, 2].imshow(pred, alpha=0.4, cmap="Reds"); axes[r, 2].set_title(f"Prediction (Dice {d:.2f})")
        for a in axes[r]:
            a.axis("off")
    fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "predictions.png"), dpi=120); plt.show()

visualize(best_model, Xte, Yte, device)

## Notes
- **Inverted-residual (+SE) encoder + attention gates + Dice loss + Adamax**: an efficient encoder that focuses on the right channels, plus spatial attention on the skips.
- Early stopping (patience 25) means training may end well before 100 epochs once val Dice plateaus - that is expected.
- Loss curve here is **Dice loss** (lower = better); it mirrors the validation Dice.
- Because LR is small, the model learns slowly - if val Dice is still rising at epoch 40, raise `EPOCHS` to 60.
- Improved slice selection (tumor + neighbour context) + float32 give the model more anatomical information.

**Memory limited?** lower `MAX_SLICES_PER_PATIENT` (e.g. 70) or set `USE_FLOAT32 = False`.
**Out of GPU memory?** lower `BATCH_SIZE` to 8.